# Preview the prepared TOC text

This notebook reads two local files:

- `data/raw/books.parquet`: the original book records, including `raw_toc_json` and `toc_entries`.
- `data/processed/toc_features.parquet`: a separate table prepared from `raw_toc_json`, with one row for every original TOC item. Each book contributes one row per TOC item, including items without text.

**What created the second file?** `bookpath.toc` is the Python module name for [src/bookpath/toc.py](../src/bookpath/toc.py): `bookpath` is the package, and `toc` is the file inside it. In that file, `build_toc_features(books)` creates the table, and `prepare_toc_features(...)` reads the raw file and saves the result. This notebook reads the saved result and, in section 6, rebuilds the expected table in memory to compare them. It does not write a processed file.

If the processed file is missing, run this command from the repository root. `-m bookpath.toc` tells Python to run that module:

```sh
uv run python -m bookpath.toc
```

**What are `base_text` and `feature_text`?** These are names we chose for **new columns in the processed table**. They are not fields supplied by the original dataset. Section 1 loads this table into the DataFrame named `toc_features`.

| New column | What the preparation code puts in it |
| --- | --- |
| `base_text` | The selected main text from the original TOC item. |
| `feature_text` | The prepared text, including a chapter subtitle when it adds information. |

Main text comes from the first nonblank `title`, otherwise `value`, otherwise `label`; an item that is already a string supplies its text directly. When there is no additional chapter subtitle, `feature_text` equals `base_text`. A subtitle can also supply text by itself when main text is absent.

Choose a book in section 2, either by its `parent_asin` or with a repeatable random selection. Sections 2–3 follow that book; sections 4–6 check the whole catalog. The conclusions are calculated from the displayed results. Both new columns contain ordinary text; no embedding vectors have been created. The separate Amazon `features` column is explored in [notebook 04](04_description_features_eda.ipynb).

## 1. Load the raw books and the processed TOC rows

Objective: Read the two saved files and record both file checksums before inspection.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from bookpath.local_data import read_books_from_parquet

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the Bookpath repository.")
    project_root = project_root.parent

raw_path = project_root / "data/raw/books.parquet"
feature_path = project_root / "data/processed/toc_features.parquet"
if not feature_path.is_file():
    raise FileNotFoundError("Run uv run python -m bookpath.toc from the repository root first.")


def file_checksum(path):
    with path.open("rb") as saved_file:
        return hashlib.file_digest(saved_file, "sha256").hexdigest()


checksum_before = file_checksum(raw_path)
feature_checksum_before = file_checksum(feature_path)
books = read_books_from_parquet(raw_path)
toc_features = pd.read_parquet(feature_path)
print(f"Raw book rows: {len(books):,}")
print(f"Processed TOC table: {len(toc_features):,} rows × {len(toc_features.columns):,} columns.")
print("Feature version:", toc_features["feature_version"].unique().tolist())

Raw book rows: 9,034
Processed TOC table: 141,513 rows × 21 columns.
Feature version: ['toc-v1']


**Where do the processed columns come from?** `toc_features` is the table loaded from `data/processed/toc_features.parquet`. Its column names are defined by the preparation code in `src/bookpath/toc.py`.

`build_toc_features(books)` creates **20 columns**: identifiers, prepared text, original TOC details, review flags, and a version label. The saving function, `prepare_toc_features(...)`, adds **`source_sha256`**, the raw file's checksum. That makes **21 saved columns**.

Some values are copied from book columns, some come from items inside `raw_toc_json`, and others are calculated. The table below lists **every column in the processed file**, grouped by purpose. These groups describe the columns; they are not additional columns themselves.

Within the original-item group, `page` comes from `pagenum`, and `entry_class` comes from `class`. `authors_json` and `description_json` store details from that TOC item; `raw_item_json` keeps the whole original item. They do not refer to the book-level Amazon `description` column.

In [2]:
column_groups = {
    "Copied book identifiers and title": ["parent_asin", "ol_edition_key", "book_title"],
    "Selected and prepared TOC text": [
        "text_source", "base_text", "chapter_subtitle", "feature_text", "subtitle_added",
    ],
    "Original TOC item and its details": [
        "raw_position", "label", "level", "page", "entry_class",
        "authors_json", "description_json", "raw_item_json",
    ],
    "Calculated review flags": ["missing_text", "numeric_only_text", "repeated_text"],
    "Preparation version and source checksum": ["feature_version", "source_sha256"],
}
listed_columns = [column for columns in column_groups.values() for column in columns]
assert sorted(listed_columns) == sorted(toc_features.columns), "Update the inventory to match the saved columns."
column_inventory = pd.DataFrame([
    {"Purpose": purpose, "Columns in toc_features": ", ".join(columns), "Number of columns": len(columns)}
    for purpose, columns in column_groups.items()
]).set_index("Purpose")
column_inventory.loc["Total saved columns"] = ["All columns listed above", len(toc_features.columns)]
with pd.option_context("display.max_colwidth", None):
    display(column_inventory)

display(Markdown(
    f"**Conclusion:** We loaded **{len(books):,} book records** and a separate prepared TOC table "
    f"with **{len(toc_features):,} rows and {len(toc_features.columns):,} columns**. "
    "Each prepared row describes one original TOC item using the columns listed above. "
    "Section 5 checks item preservation; section 6 checks every saved column against the code."
))

,Columns in toc_features,Number of columns
Purpose,,
Copied book identifiers and title,"parent_asin, ol_edition_key, book_title",3
Selected and prepared TOC text,"text_source, base_text, chapter_subtitle, feature_text, subtitle_added",5
Original TOC item and its details,"raw_position, label, level, page, entry_class, authors_json, description_json, raw_item_json",8
Calculated review flags,"missing_text, numeric_only_text, repeated_text",3
Preparation version and source checksum,"feature_version, source_sha256",2
Total saved columns,All columns listed above,21


**Conclusion:** We loaded **9,034 book records** and a separate prepared TOC table with **141,513 rows and 21 columns**. Each prepared row describes one original TOC item using the columns listed above. Section 5 checks item preservation; section 6 checks every saved column against the code.

## 2. Compare the same TOC item in three places

**Objective:** Choose a book, then compare one item in its `raw_toc_json`, `toc_entries`, and prepared TOC table.

**Try another book:**

- Leave `example_asin = None` to pick a random book. `random_seed = 42` makes the choice repeatable; change the seed to pick another example.
- To choose a specific book, replace `None` with its quoted ID, for example `example_asin = "0142000280"` for *Getting Things Done*.
- After changing either setting, rerun this section and section 3. The selected ID and title appear below so you can return to the same example later.

The comparison uses the **first original item with main text** and the **first saved parsed entry**. Original items without main text were omitted from `toc_entries`, so their positions can differ. We show the original position explicitly; all original rows remain visible in section 3. This one-book example does not describe the whole catalog.

In [3]:
example_asin = None  # Or enter a specific ID, such as "0142000280".
random_seed = 42     # Change this number for another repeatable random choice.

if example_asin is None:
    example_asin = books.sample(n=1, random_state=random_seed)["parent_asin"].iloc[0]

selected = books.loc[books["parent_asin"].eq(example_asin)]
if selected.empty:
    raise ValueError(f"No saved book has parent_asin = {example_asin!r}. Use an existing ID or None.")
selected_book = selected.iloc[0]
book_features = toc_features.loc[
    toc_features["parent_asin"].eq(example_asin)
].sort_values("raw_position")

print("Selected parent_asin:", example_asin)
print("Book:", selected_book["title"])

Selected parent_asin: 0631228306
Book: Carbonell Museum Studies: An Anthology of Contexts


In [4]:
items_with_main_text = book_features.loc[book_features["base_text"].notna()]
if items_with_main_text.empty:
    raise ValueError("This book has no prepared item with main text to compare. Choose another ID.")
first_feature = items_with_main_text.iloc[0]
raw_position = int(first_feature["raw_position"])
raw_entry = json.loads(selected_book["raw_toc_json"])[raw_position - 1]
existing_entry = selected_book["toc_entries"][0]

print(f"1. Original item at position {raw_position} in raw_toc_json:")
print(json.dumps(raw_entry, ensure_ascii=False, indent=2))
print("\n2. First saved entry in toc_entries:")
print(json.dumps(existing_entry, ensure_ascii=False, indent=2))
print("\n3. Prepared feature_text for that original item:")
print(first_feature["feature_text"])

1. Original item at position 1 in raw_toc_json:
{
  "level": 0,
  "title": "Introduction: Museum/studies and the \"eccentric space\" of an anthology / Bettina Messias Carbonell",
  "type": {
    "key": "/type/toc_item"
  }
}

2. First saved entry in toc_entries:
{
  "level": "0",
  "page": null,
  "sequence": "1",
  "source_key": "title",
  "text": "Introduction: Museum/studies and the \"eccentric space\" of an anthology / Bettina Messias Carbonell"
}

3. Prepared feature_text for that original item:
Introduction: Museum/studies and the "eccentric space" of an anthology / Bettina Messias Carbonell


**Where does `chapter_subtitle` come from?** It is a new column in the processed table. For each original item inside `raw_toc_json`, `toc.py` reads the item's `subtitle` and removes spaces at the start and end. A missing, blank, or non-text subtitle becomes a missing value (`<NA>`). This is the chapter's subtitle; the book-level `subtitle` column is separate.

For example, an item's `subtitle = "  Practical examples  "` becomes `chapter_subtitle = "Practical examples"`. **This column holds only the subtitle.** If `base_text` is `"Learning Python"`, the combined text goes into `feature_text`: `"Learning Python: Practical examples"`.

The code adds a subtitle with words only when those words do not already appear together in `base_text`, ignoring case and punctuation. A subtitle can supply text on its own when main text is missing. `subtitle_added = True` records that the subtitle contributed to `feature_text`; otherwise `feature_text` stays equal to `base_text`.

**What happened to our selected item?** The first table shows its original `subtitle` alongside the prepared columns. The second shows its source and other details. Missing values mean that detail was unavailable. Labels, levels, and pages are stored separately.


In [5]:
comparison = pd.DataFrame({
    "field": [
        "existing toc_entries text", "original item: subtitle", "base_text",
        "chapter_subtitle", "feature_text", "subtitle_added",
    ],
    "value": [
        existing_entry["text"],
        raw_entry.get("subtitle") if isinstance(raw_entry, dict) else None,
        first_feature["base_text"],
        first_feature["chapter_subtitle"],
        first_feature["feature_text"],
        first_feature["subtitle_added"],
    ],
})
with pd.option_context("display.max_colwidth", None):
    display(comparison)
    display(first_feature[[
        "parent_asin", "ol_edition_key", "raw_position", "text_source",
        "label", "level", "page", "entry_class",
    ]].rename("value").to_frame())

text_agrees = first_feature["base_text"] == existing_entry["text"]
text_result = "matches" if text_agrees else "does not match"
if first_feature["subtitle_added"]:
    subtitle_result = "The chapter subtitle adds text to feature_text."
elif pd.isna(first_feature["chapter_subtitle"]):
    subtitle_result = "This item has no chapter subtitle, so feature_text equals base_text."
else:
    subtitle_result = "The chapter subtitle adds no text under the preparation rule; feature_text equals base_text."

display(Markdown(
    f"**Conclusion:** For original item **{raw_position}** in book `{example_asin}`, "
    f"`base_text` **{text_result}** the first saved parsed entry. {subtitle_result} "
    "The original position lets us trace this prepared row back to its source."
))

,field,value
0,existing toc_entries text,"Introduction: Museum/studies and the ""eccentric space"" of an anthology / Bettina Messias Carbonell"
1,original item: subtitle,None
2,base_text,"Introduction: Museum/studies and the ""eccentric space"" of an anthology / Bettina Messias Carbonell"
3,chapter_subtitle,<NA>
4,feature_text,"Introduction: Museum/studies and the ""eccentric space"" of an anthology / Bettina Messias Carbonell"
5,subtitle_added,False


,value
parent_asin,0631228306
ol_edition_key,/books/OL23240713M
raw_position,1
text_source,title
label,<NA>
level,0
page,<NA>
entry_class,<NA>


**Conclusion:** For original item **1** in book `0631228306`, `base_text` **matches** the first saved parsed entry. This item has no chapter subtitle, so feature_text equals base_text. The original position lets us trace this prepared row back to its source.

## 3. Read every prepared TOC item for this book

**Objective:** Inspect all items for the book chosen in section 2, in their original order. The table includes supporting sections and any rows without text, as well as chapters.

`subtitle_added` identifies items that received subtitle text. `missing_text` identifies items with no prepared text. Changing the selected book changes this table and its conclusion.

In [6]:
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(book_features[[
        "raw_position", "feature_text", "subtitle_added", "missing_text",
    ]].set_index("raw_position"))

display(Markdown(
    f"**Conclusion:** Book `{example_asin}` has **{len(book_features):,} prepared TOC rows**. "
    f"**{int(book_features['subtitle_added'].sum()):,}** received chapter subtitle text, and "
    f"**{int(book_features['missing_text'].sum()):,}** have no prepared text. "
    "The rows are displayed in original TOC order, including any empty items."
))

,feature_text,subtitle_added,missing_text
raw_position,,,
1,"Introduction: Museum/studies and the ""eccentric space"" of an anthology / Bettina Messias Carbonell",False,False
2,From The museum age : foreword / Germain Bazin,False,False
3,The museum : its classical etymology and renaissance genealogy / Paula Findlen,False,False
4,The universal survey museum / Carol Duncan and Alan Wallach,False,False
5,Brain of the earth's body : museums and the framing of modernity / Donald Preziosi,False,False
6,The museum refuses to stand still / Kenneth Hudson,False,False
7,"The mirror and the tomb : Africa, museums, and memory / Françoise Lionnet",False,False
8,Seeing through solidity : a feminist perspective on museums / Gaby Porter,False,False
9,"Museums of ""human suffering"" and the struggle for human rights / Terence M. Duffy",False,False


**Conclusion:** Book `0631228306` has **54 prepared TOC rows**. **0** received chapter subtitle text, and **0** have no prepared text. The rows are displayed in original TOC order, including any empty items.

## 4. Inspect TOC items marked for review

**Objective:** Count and inspect prepared items that have no text, contain only digits, or repeat within the same book.

A **flag** is a `True`/`False` column created by `toc.py`. `True` means the row meets the condition below. A **flagged item** has at least one of these columns set to `True`.

| Flag column | When it is `True` |
| --- | --- |
| `missing_text` | `feature_text` is missing after trying the main-text fields and chapter subtitle. |
| `numeric_only_text` | `feature_text` contains only digits, such as `"123"`. |
| `repeated_text` | The same `feature_text` appears more than once within this book, ignoring case and repeated whitespace. Every occurrence is marked. |

**These rows were processed by `toc.py`.** It prepared their text where available, kept their original item and position, and calculated the flags. The rows remain in `data/processed/toc_features.parquet` so we can trace and inspect them. Missing text is left missing; a number or repeated heading may be legitimate, so a flag alone does not justify deleting an item. These flags let us consider filtering later without losing the source information.

This section covers **all book records**, regardless of the selection in section 2. The code below displays **five tables: two summaries, followed by three sets of examples**.

- **Table 1 — Flag counts:** Totals the flag matches. An item with two flags contributes twice.
- **Table 2 — Item counts:** Counts each item once: at least one flag, no flags, and the total. The first two rows add up to the total.
- **Tables 3–5 — Examples:** Show up to five items with missing text, numeric-only text, and repeated text, respectively. These are the first matching rows in saved order, already included in the summaries.

Adding a chapter subtitle is a separate preparation action. Its count is shown separately and is not added to either flag total.


In [7]:
review_columns = ["missing_text", "numeric_only_text", "repeated_text"]
print(f"Chapter subtitle added: {int(toc_features['subtitle_added'].sum()):,} of {len(toc_features):,} TOC items.")

flag_counts = toc_features[review_columns].sum().astype(int)
flag_matches = int(flag_counts.sum())
flag_counts.loc["Total flag matches (overlaps counted)"] = flag_matches
display(Markdown("**Table 1 of 5 — Flag counts (an item can contribute to multiple flags)**"))
display(flag_counts.rename_axis("Review flag").rename("Count").to_frame())

has_review_flag = toc_features[review_columns].any(axis=1)
flagged_items = int(has_review_flag.sum())
item_totals = pd.Series({
    "At least one flag (each item counted once)": flagged_items,
    "No flags": int((~has_review_flag).sum()),
    "Total TOC items": len(toc_features),
}, name="TOC items").to_frame()
display(Markdown("**Table 2 of 5 — Item counts (each item counted once)**"))
display(item_totals)

# Number the example tables 3, 4, and 5.
for table_number, flag in enumerate(review_columns, start=3):
    display(Markdown(f"**Table {table_number} of 5 — Up to five examples where `{flag}` is True**"))
    display(toc_features.loc[toc_features[flag], [
        "parent_asin", "raw_position", "base_text", "feature_text",
    ]].head(5))

display(Markdown(
    f"**Conclusion:** **{flag_matches:,} flag matches** affect **{flagged_items:,} distinct TOC items**. "
    f"The other **{len(toc_features) - flagged_items:,} items** have none of these three flags, "
    f"giving **{len(toc_features):,} items in total**. Flagged rows remain in the prepared table; "
    "a flag alone does not establish an error."
))

Chapter subtitle added: 16 of 141,513 TOC items.


**Table 1 of 5 — Flag counts (an item can contribute to multiple flags)**

,Count
Review flag,
missing_text,18
numeric_only_text,435
repeated_text,2916
Total flag matches (overlaps counted),3369


**Table 2 of 5 — Item counts (each item counted once)**

,TOC items
At least one flag (each item counted once),3361
No flags,138152
Total TOC items,141513


...

**Table 4 of 5 — Up to five examples where `numeric_only_text` is True**

,parent_asin,raw_position,base_text,feature_text
1708,0062385801,1,1,1
1721,0062385801,14,351,351
1722,0062385801,15,353,353
1723,0062385801,16,357,357
1724,0062385801,17,375,375


**Table 5 of 5 — Up to five examples where `repeated_text` is True**

,parent_asin,raw_position,base_text,feature_text
338,0060566582,6,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...
339,0060566582,7,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...
340,0060566582,8,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...
341,0060566582,9,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...
342,0060566582,10,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...,Cal Thomas -- pt. 1 -- pt. II -- pt. III -- pt...


**Conclusion:** **3,369 flag matches** affect **3,361 distinct TOC items**. The other **138,152 items** have none of these three flags, giving **141,513 items in total**. Flagged rows remain in the prepared table; a flag alone does not establish an error.

## 5. Validate text agreement and item preservation

**Objective:** Run two **data preservation checks** for every book, using `parent_asin` to connect the original book record with its prepared TOC rows.

| Check in the output | What we compare | What a match confirms |
| --- | --- | --- |
| `existing_text_matches` | Non-missing `base_text` values in `toc_features`, against the `text` values inside that book's saved `toc_entries` list. | The main-text lists contain exactly the same text in the same order. |
| `raw_positions_preserved` | `raw_position` values in `toc_features`, against positions 1 through the number of items in that book's `raw_toc_json` list. | Every original position is present once, in order, including positions of items without text. |

The output counts **books passing each check**, not TOC items. Both checks cover every book in this saved dataset; a failed check stops the notebook at `assert`.

This validates text agreement and item preservation. Section 6 checks all prepared columns against the code. Neither check establishes that the source TOC is complete or belongs to the correct book, or that recommendations will be useful.


In [8]:
base_text_by_book = (
    toc_features.loc[toc_features["base_text"].notna()]
    .groupby("parent_asin")["base_text"].agg(list)
)
raw_positions_by_book = toc_features.groupby("parent_asin")["raw_position"].agg(list)
validation_rows = []
for book in books.itertuples():
    expected_text = [entry["text"] for entry in book.toc_entries]
    raw_count = len(json.loads(book.raw_toc_json))
    validation_rows.append({
        "parent_asin": book.parent_asin,
        "existing_text_matches": base_text_by_book.get(book.parent_asin, []) == expected_text,
        "raw_positions_preserved": raw_positions_by_book.get(book.parent_asin, []) == list(range(1, raw_count + 1)),
    })

validation = pd.DataFrame(validation_rows)
check_columns = ["existing_text_matches", "raw_positions_preserved"]
display(validation[check_columns].sum().rename("matching_books").to_frame())
assert validation[check_columns].all().all(), "Inspect books that failed the comparison."

display(Markdown(
    f"**Conclusion:** All **{len(validation):,} book records** pass both checks. "
    f"Their **{int(toc_features['base_text'].notna().sum()):,} main-text values** match "
    "the existing parsed text in order, and every original item position is retained. "
    "These checks confirm main-text agreement and preservation of original positions. "
    "They do not establish that each source TOC is complete or correctly matched to its book."
))

,matching_books
existing_text_matches,9034
raw_positions_preserved,9034


**Conclusion:** All **9,034 book records** pass both checks. Their **141,495 main-text values** match the existing parsed text in order, and every original item position is retained. These checks confirm main-text agreement and preservation of original positions. They do not establish that each source TOC is complete or correctly matched to its book.

## 6. Does the saved table match its source and the current code?

**Objective:** Recreate the prepared table in memory and compare every row and column with the saved file. The column count below comes from `len(toc_features.columns)`: the **21 processed columns listed in section 1**.

Section 5 checked the main text and item positions. Here we also check `feature_text`, chapter subtitles, metadata, review flags, and the preparation version. This detects a saved output that no longer matches the current preparation code, even when the raw file has not changed.

`build_toc_features(books)` creates the expected table using the already-loaded books. We add the raw file's checksum as `source_sha256`, just as the saving step does. `assert_frame_equal()` checks that both tables have the same rows, columns, values, data types, and order. A difference stops the notebook with an error describing the mismatch; the success conclusion appears only if every check passes.

Everything is calculated in memory. The final checksum checks confirm that **neither saved file changed**. After editing `toc.py`, restart the kernel and run all cells so this comparison uses the updated code.

In [ ]:
from bookpath.toc import build_toc_features

assert toc_features["source_sha256"].eq(checksum_before).all()
assert not toc_features.duplicated(["parent_asin", "raw_position"]).any()

rebuilt_features = build_toc_features(books)
rebuilt_features["source_sha256"] = checksum_before
pd.testing.assert_frame_equal(toc_features, rebuilt_features, check_exact=True)
print(f"Full comparison passed: {len(toc_features):,} TOC rows and {len(toc_features.columns):,} processed columns (listed in section 1).")

assert file_checksum(raw_path) == checksum_before, "The raw Parquet changed during inspection."
assert file_checksum(feature_path) == feature_checksum_before, "The processed Parquet changed during inspection."
print(f"Source checksum: {checksum_before}")
print("Both saved files are unchanged.")

display(Markdown(
    f"**Conclusion:** The saved `toc_features` table has **{len(toc_features):,} TOC-item rows** "
    f"and **{len(toc_features.columns):,} columns**, listed in section 1. Every row and column "
    "matches the current preparation code, including the enriched text, metadata, review flags, and version. "
    "Each book/item-position pair is unique, and both saved files are unchanged. "
    "This confirms that the saved output agrees with the code; recommendation quality still needs evaluation."
))

## What we take into the first baseline

The TOC preparation is ready for a first retrieval baseline. When we build that pipeline, we will use these decisions:

| Decision | What we will use |
| --- | --- |
| Input file | `data/processed/toc_features.parquet`, created by [src/bookpath/toc.py](../src/bookpath/toc.py). |
| Text and identifiers | Use `feature_text`; keep `parent_asin` and `raw_position` to identify the book and original TOC item. |
| Missing text | Keep these rows in the prepared file. Skip missing or blank `feature_text` when creating embeddings. |
| Numeric and repeated text | Keep these items initially, together with their review flags. Evaluate whether filtering improves retrieval before changing that choice. |

These are decisions for the next pipeline step; this notebook creates no embeddings. The checks above confirm that the saved preparation matches the code. Retrieval evaluation will tell us whether the prepared text helps recommendations.

**Next:** Review the book-level `description` and `features` columns in [notebook 04](04_description_features_eda.ipynb), then build and evaluate the first retrieval baseline.
